# 02 - Location & Sector Hierarchy (Double-Counting Guards)

Classify locations (state/region/national) and verify sector aggregates so we never double-count when summing.

In [1]:
import sys, os
ROOT = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), "src")) else os.path.dirname(os.getcwd())
sys.path.insert(0, ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
DATA = os.path.join(ROOT, "data", "eia_coal_consumption_quality_2020_2025.csv")
CHARTS = os.path.join(ROOT, "charts")
os.makedirs(CHARTS, exist_ok=True)
print("ROOT:", ROOT)


ROOT: C:\Users\Aditi\Documents\Projects\BDAA&SIG_hack_Fall26


In [2]:
from src.data import load_coal_data, CLEAN_SECTOR_BREAKDOWN, SECTOR_LEAVES
from src.taxonomy import (add_location_level, filter_location, total_coal_use,
    check_sector_aggregates, check_us_vs_states, check_regions_vs_states,
    REGION_CODES, REGION_MEMBERS)
df = load_coal_data(DATA)
df = add_location_level(df)
print('Location level counts:')
print(df['location_level'].value_counts())
print('\nRegion codes:', sorted(REGION_CODES))

Location level counts:
location_level
state       1291
region       411
national      48
Name: count, dtype: int64

Region codes: ['90', 'ENC', 'ESC', 'MAT', 'MTN', 'NEW', 'PCC', 'PCN', 'SAT', 'WNC', 'WSC']


In [3]:
# Critical guard: sector aggregates 98==1+2+3 and 94==2+3 everywhere components present.
agg_fails = check_sector_aggregates(df, tol=1.0)
print('Sector-aggregate mismatches:', len(agg_fails))
assert agg_fails.empty, 'sector aggregates do not reconcile'
print('Sector aggregates reconcile exactly: 98==1+2+3 and 94==2+3.')

Sector-aggregate mismatches: 0
Sector aggregates reconcile exactly: 98==1+2+3 and 94==2+3.


In [4]:
# US Total vs sum of states (leaf sectors).
usrep = check_us_vs_states(df)
print(usrep[usrep['pct_diff'].abs() > 0.01].to_string(index=False))
print('\nMax abs pct diff:', round(usrep['pct_diff'].abs().max(), 2), '%')

 period  sector    us_total  states_sum  pct_diff
   2020       2 101818321.0 103318495.0  1.473383
   2021       2 119838298.0 121469593.0  1.361247
   2022       2 114789813.0 116306018.0  1.320853
   2023       2  89325222.0  90669870.0  1.505340
   2024       2  82568292.0  84026408.0  1.765952
   2025       2  93673987.0  95070830.0  1.491175

Max abs pct diff: 1.77 %


In [5]:
# Region vs member-state sums (document any membership gaps).
regfails = check_regions_vs_states(df, tol_pct=0.5)
print('Region-vs-state mismatches (>0.5%):', len(regfails))
if not regfails.empty:
    print(regfails.head(20).to_string(index=False))

Region-vs-state mismatches (>0.5%):

 18
region  period  sector  region_value  states_sum  pct_diff
   SAT    2020       2     5588268.0   7088442.0     26.85
   SAT    2020      94     6341315.0   7841489.0     23.66
   SAT    2020      98    53093538.0  54593712.0      2.83
   SAT    2021       2     7591909.0   9223204.0     21.49
   SAT    2021      94     8406252.0  10037547.0     19.41
   SAT    2021      98    61523095.0  63154390.0      2.65
   SAT    2022       2     7071662.0   8587867.0     21.44
   SAT    2022      94     7923796.0   9440001.0     19.13
   SAT    2022      98    53138341.0  54654546.0      2.85
   SAT    2023       2     4447779.0   5792427.0     30.23
   SAT    2023      94     4903720.0   6248368.0     27.42
   SAT    2023      98    46950069.0  48294717.0      2.86
   SAT    2024       2     3777492.0   5235608.0     38.60
   SAT    2024      94     3956712.0   5414828.0     36.85
   SAT    2024      98    47893740.0  49351856.0      3.04
   SAT    2025       2     3339154.0   4735997.0    

In [6]:
# Clean non-overlapping total U.S. coal use = sectors 98 + 8 + 9 + 10 at US level.
total = total_coal_use(df, level='national')
print('Total U.S. coal use (thousand short tons), clean breakdown:')
print(total.round(0).to_string())

Total U.S. coal use (thousand short tons), clean breakdown:
period
2020    476693102.0
2021    545679285.0
2022    515534183.0
2023    425894358.0
2024    410938686.0
2025    450779790.0


**Takeaways**
- Sector aggregates reconcile exactly: `98 = 1+2+3`, `94 = 2+3`.
- U.S. Total equals the sum of states for every leaf sector **except** sector 2 (IPP Non-CHP), where the state sum exceeds the U.S. total by ~1.3-1.8% (a small EIA reporting quirk; we use U.S. rows directly for national trends).
- Clean, non-overlapping end-use breakdown: **Electric Power (98) + Commercial & Institutional (8) + Coke Plants (9) + Other Industrial (10)**.